# Stacking Ensemble — Fake News Classifier

**Task:** Binary classification of political statements as true (0) or false (1).

## What is Stacking?

Stacking (stacked generalisation) is a two-level ensemble:

1. **Level 0 — Base models**: several diverse classifiers each trained with 5-fold CV, producing Out-of-Fold (OOF) probability predictions.
2. **Level 1 — Meta-learner**: a simple Logistic Regression trained on the stacked OOF matrix. It learns *which base models to trust* for each type of sample.

The key insight is that the OOF probabilities are **unbiased** — each prediction came from a model that never saw that sample during training. This lets the meta-learner train without overfitting.

```
Raw features ──→ XGBoost   ──→ OOF proba₁ ──┐
             ──→ RFC       ──→ OOF proba₂ ──┤
             ──→ LightGBM  ──→ OOF proba₃ ──┤──→ meta-LR ──→ final proba
             ──→ CatBoost  ──→ OOF proba₄ ──┘
```

## Architecture

- **Preprocessing**: mirrors CatBoost-optB — `all-mpnet-base-v2` embeddings (768-dim, stronger than MiniLM), NER features, interaction keys, OrdinalEncoder
- **Base models** (4, no inner HP search — best-known HPs from prior experiments):
  - **XGBoost** (replaced base LR from Experiment A/B/C — LR had the lowest AUC at 0.6720)
  - **Random Forest** (n_estimators=500, max_features=0.3)
  - **LightGBM** (optB config — reg_alpha/reg_lambda tuned)
  - **CatBoost** (optB config — speaker true-rate dropped)
- **True-rate features**: `drop_speaker_true_rate=True` — confirmed best across individual models and stacking
- **Meta-learner**: `LogisticRegression(C=0.1)` on the 4 stacked OOF probabilities (no class_weight — base probas already encode the imbalance)
- **Optional late fusion**: if transformer OOF artifacts exist, adds a 5th base model column
- **Threshold tuning** on meta-LR OOF probabilities

**Metric:** Macro F1 (primary).

## Imports and Project Setup

In [ ]:
from datetime import datetime
from pathlib import Path
import sys
from time import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'train.csv').exists() and (candidate / 'src').exists():
            return candidate
    raise FileNotFoundError('Could not locate the project root.')

project_root = find_project_root(Path.cwd())
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from preprocessing.one_step import OneStepOptions, preprocess_one_step

print(f'Project root: {project_root}')

_script_start = time()
def _now() -> str:
    return datetime.now().strftime('%H:%M:%S')

## Data Loading

In [ ]:
df = pd.read_csv(project_root / 'data' / 'train.csv')
print(f'Shape: {df.shape}')
print(f"\nLabel distribution:\n{df['label'].value_counts(normalize=True).round(4)}")
df.head()

## Preprocessing Configuration

A single shared preprocessing pipeline is used for all four base models — this is a deliberate design choice:
- All base models see the same input features, so any performance difference reflects the model's inductive bias, not a data preparation advantage.
- The pipeline mirrors the CatBoost-optB config (best individual model).

**Key upgrade from individual models:** `statement_embedding_model = 'all-mpnet-base-v2'` (768-dim) instead of `all-MiniLM-L6-v2` (384-dim). MPNet produces richer semantic representations at the cost of 2× the vector size.

In [ ]:
label_option      = 'skip'
label_source_col  = 'label'
id_option         = 'drop'

subject_source_col             = 'subject'
subject_keep_original          = False
subject_clean_text             = True
subject_normalize_separators   = True
subject_split_topics           = True
subject_primary_strategy       = 'most_frequent'
subject_rare_threshold         = 10
subject_rare_label             = 'other'
subject_max_topics_for_primary = None
subject_multi_topic_label      = 'multi-topic'
subject_add_primary            = True
subject_add_topic_count        = True
subject_add_multiple_topics_flag = True
subject_add_topic_list         = False
subject_add_length_features    = True
subject_add_grouped_primary    = True
subject_group_rare             = True
subject_add_subject_frequency  = True
subject_add_subject_is_rare    = True
subject_add_subject_primary_true_rate = False
subject_label_col              = None
subject_scale                  = 'none'
subject_verbose                = False

In [ ]:
statement_source_col             = 'statement'
statement_original_output_col    = 'statement_original'
statement_output_col             = 'statement_clean'
statement_keep_original          = False
statement_lower                  = True
statement_remove_html            = True
statement_remove_urls            = True
statement_replace_numbers        = False
statement_number_token           = '<NUM>'
statement_stopword_removal       = False
statement_keep_negations         = True
statement_remove_punctuation     = False
statement_stemmer                = 'none'
statement_lemmatizer             = 'none'
statement_repair_polluted_statements = True
statement_add_rare_token_features   = True
statement_rare_token_threshold      = 1
statement_token_freqs               = None
statement_add_spelling_errors       = True
statement_add_lexical_features      = True
statement_add_pollution_features    = True
statement_add_ner_features          = True
statement_ner_model                 = 'en_core_web_sm'
statement_vectorizer_type           = 'embeddings'
statement_vectorizer_max_features   = 500
statement_vectorizer_min_df         = 2
statement_vectorizer_max_df         = 0.7
# Upgraded to MPNet (768-dim) — richer semantic representations than MiniLM (384-dim)
statement_embedding_model           = 'all-mpnet-base-v2'
statement_fitted_vectorizer         = None
statement_scale                     = 'none'
statement_verbose                   = False

In [ ]:
speaker_source_col            = 'speaker'
speaker_keep_original         = False
speaker_clean_text            = True
speaker_normalize_separators  = True
speaker_group_rare            = True
speaker_rare_threshold        = 5
speaker_rare_label            = 'other'
speaker_add_frequency         = True
speaker_add_is_rare           = True
speaker_add_grouped_speaker   = True
speaker_add_length_features   = True
speaker_add_title_flag        = True
speaker_add_comma_flag        = True
speaker_add_period_flag       = True
speaker_add_token_count       = False
speaker_add_speaker_primary_true_rate = False
speaker_label_col             = None
speaker_scale                 = 'none'
speaker_verbose               = False

speaker_job_source_col           = 'speaker_job'
speaker_job_keep_original        = False
speaker_job_clean_text           = True
speaker_job_normalize_separators = True
speaker_job_group_rare           = True
speaker_job_rare_threshold       = 5
speaker_job_rare_label           = 'other'
speaker_job_add_frequency        = True
speaker_job_add_is_rare          = True
speaker_job_add_grouped_job      = True
speaker_job_add_length_features  = True
speaker_job_add_title_flag       = True
speaker_job_add_comma_flag       = True
speaker_job_add_slash_flag       = True
speaker_job_add_ampersand_flag   = True
speaker_job_add_token_count      = False
speaker_job_add_job_primary_true_rate = False
speaker_job_job_label_col        = None
speaker_job_scale                = 'none'
speaker_job_verbose              = False

party_affiliation_source_col         = 'party_affiliation'
party_affiliation_keep_original      = False
party_affiliation_clean_text         = True
party_affiliation_group_rare         = True
party_affiliation_rare_threshold     = 5
party_affiliation_rare_label         = 'other'
party_affiliation_add_frequency      = True
party_affiliation_add_is_rare        = True
party_affiliation_add_grouped_party  = True
party_affiliation_add_length_features = True
party_affiliation_add_token_count    = True
party_affiliation_add_is_major_party = True
party_affiliation_add_is_institutional = True
party_affiliation_add_slash_flag     = True
party_affiliation_add_ampersand_flag = True
party_affiliation_add_comma_flag     = False
party_affiliation_add_parentheses_flag = True
party_affiliation_add_party_primary_true_rate = False
party_affiliation_party_label_col    = None
party_affiliation_scale              = 'none'
party_affiliation_verbose            = False

state_source_col        = 'state_info'
state_drop              = False
state_keep_original     = False
state_clean_text        = True
state_normalize_state   = True
state_group_rare        = True
state_rare_threshold    = 5
state_rare_label        = 'other'
state_add_is_us_state   = True
state_add_frequency     = True
state_add_is_rare       = True
state_add_grouped_state = True
state_add_has_us_words  = True
state_add_us_region     = True
state_add_length_features = False
state_add_token_count   = True
state_scale             = 'none'
state_verbose           = False

fe_statement_col          = 'statement_clean'
fe_statement_original_col = 'statement_original'
fe_speaker_col            = 'speaker_clean'
fe_subject_col            = 'subject_clean'
fe_party_col              = 'party_affiliation_clean'
fe_speaker_job_col        = 'speaker_job_clean'
fe_state_col              = 'state_info_clean'
fe_label_col              = None
fe_add_speaker_subject              = True
fe_add_speaker_party                = True
fe_add_subject_party                = True
fe_add_speaker_job_subject          = True
fe_add_state_party                  = True
fe_add_speaker_statement_len_bucket = True
fe_statement_len_bins               = (50, 150)
fe_add_speaker_true_rate            = False
fe_add_subject_true_rate            = False
fe_add_party_true_rate              = False
fe_add_speaker_avg_statement_len    = True
fe_add_subject_avg_statement_len    = True
fe_add_speaker_avg_punctuation      = True
fe_add_speaker_avg_number_ratio     = True
fe_add_negation_count    = True
fe_add_hedge_count       = True
fe_add_absolutist_count  = True
fe_add_numeral_count     = True
fe_add_proper_noun_count = False
fe_add_readability       = True
fe_add_sentiment         = True
fe_scale                 = 'none'
fe_verbose               = False

In [ ]:
options = OneStepOptions(
    label_option=label_option, label_source_col=label_source_col, id_option=id_option,
    subject_source_col=subject_source_col, subject_keep_original=subject_keep_original,
    subject_clean_text=subject_clean_text, subject_normalize_separators=subject_normalize_separators,
    subject_split_topics=subject_split_topics, subject_primary_strategy=subject_primary_strategy,
    subject_max_topics_for_primary=subject_max_topics_for_primary,
    subject_multi_topic_label=subject_multi_topic_label,
    subject_add_primary=subject_add_primary, subject_add_topic_count=subject_add_topic_count,
    subject_add_multiple_topics_flag=subject_add_multiple_topics_flag,
    subject_add_topic_list=subject_add_topic_list, subject_add_length_features=subject_add_length_features,
    subject_add_grouped_primary=subject_add_grouped_primary, subject_group_rare=subject_group_rare,
    subject_rare_threshold=subject_rare_threshold, subject_rare_label=subject_rare_label,
    subject_add_subject_frequency=subject_add_subject_frequency,
    subject_add_subject_is_rare=subject_add_subject_is_rare,
    subject_add_subject_primary_true_rate=subject_add_subject_primary_true_rate,
    subject_label_col=subject_label_col, subject_scale=subject_scale, subject_verbose=subject_verbose,
    statement_source_col=statement_source_col, statement_original_output_col=statement_original_output_col,
    statement_output_col=statement_output_col, statement_keep_original=statement_keep_original,
    statement_lower=statement_lower, statement_remove_html=statement_remove_html,
    statement_remove_urls=statement_remove_urls, statement_replace_numbers=statement_replace_numbers,
    statement_number_token=statement_number_token, statement_stopword_removal=statement_stopword_removal,
    statement_keep_negations=statement_keep_negations, statement_stemmer=statement_stemmer,
    statement_lemmatizer=statement_lemmatizer, statement_remove_punctuation=statement_remove_punctuation,
    statement_repair_polluted_statements=statement_repair_polluted_statements,
    statement_add_rare_token_features=statement_add_rare_token_features,
    statement_rare_token_threshold=statement_rare_token_threshold,
    statement_token_freqs=statement_token_freqs, statement_add_spelling_errors=statement_add_spelling_errors,
    statement_add_lexical_features=statement_add_lexical_features,
    statement_add_pollution_features=statement_add_pollution_features,
    statement_add_ner_features=statement_add_ner_features, statement_ner_model=statement_ner_model,
    statement_vectorizer_type=statement_vectorizer_type,
    statement_vectorizer_max_features=statement_vectorizer_max_features,
    statement_vectorizer_min_df=statement_vectorizer_min_df,
    statement_vectorizer_max_df=statement_vectorizer_max_df,
    statement_embedding_model=statement_embedding_model,
    statement_fitted_vectorizer=statement_fitted_vectorizer,
    statement_scale=statement_scale, statement_verbose=statement_verbose,
    speaker_source_col=speaker_source_col, speaker_keep_original=speaker_keep_original,
    speaker_clean_text=speaker_clean_text, speaker_normalize_separators=speaker_normalize_separators,
    speaker_group_rare=speaker_group_rare, speaker_rare_threshold=speaker_rare_threshold,
    speaker_rare_label=speaker_rare_label, speaker_add_frequency=speaker_add_frequency,
    speaker_add_is_rare=speaker_add_is_rare, speaker_add_grouped_speaker=speaker_add_grouped_speaker,
    speaker_add_length_features=speaker_add_length_features, speaker_add_title_flag=speaker_add_title_flag,
    speaker_add_comma_flag=speaker_add_comma_flag, speaker_add_period_flag=speaker_add_period_flag,
    speaker_add_token_count=speaker_add_token_count,
    speaker_add_speaker_primary_true_rate=speaker_add_speaker_primary_true_rate,
    speaker_label_col=speaker_label_col, speaker_scale=speaker_scale, speaker_verbose=speaker_verbose,
    speaker_job_source_col=speaker_job_source_col, speaker_job_keep_original=speaker_job_keep_original,
    speaker_job_clean_text=speaker_job_clean_text,
    speaker_job_normalize_separators=speaker_job_normalize_separators,
    speaker_job_group_rare=speaker_job_group_rare, speaker_job_rare_threshold=speaker_job_rare_threshold,
    speaker_job_rare_label=speaker_job_rare_label, speaker_job_add_frequency=speaker_job_add_frequency,
    speaker_job_add_is_rare=speaker_job_add_is_rare, speaker_job_add_grouped_job=speaker_job_add_grouped_job,
    speaker_job_add_length_features=speaker_job_add_length_features,
    speaker_job_add_title_flag=speaker_job_add_title_flag,
    speaker_job_add_comma_flag=speaker_job_add_comma_flag, speaker_job_add_slash_flag=speaker_job_add_slash_flag,
    speaker_job_add_ampersand_flag=speaker_job_add_ampersand_flag,
    speaker_job_add_token_count=speaker_job_add_token_count,
    speaker_job_add_job_primary_true_rate=speaker_job_add_job_primary_true_rate,
    speaker_job_job_label_col=speaker_job_job_label_col,
    speaker_job_scale=speaker_job_scale, speaker_job_verbose=speaker_job_verbose,
    party_affiliation_source_col=party_affiliation_source_col,
    party_affiliation_keep_original=party_affiliation_keep_original,
    party_affiliation_clean_text=party_affiliation_clean_text,
    party_affiliation_group_rare=party_affiliation_group_rare,
    party_affiliation_rare_threshold=party_affiliation_rare_threshold,
    party_affiliation_rare_label=party_affiliation_rare_label,
    party_affiliation_add_frequency=party_affiliation_add_frequency,
    party_affiliation_add_is_rare=party_affiliation_add_is_rare,
    party_affiliation_add_grouped_party=party_affiliation_add_grouped_party,
    party_affiliation_add_length_features=party_affiliation_add_length_features,
    party_affiliation_add_token_count=party_affiliation_add_token_count,
    party_affiliation_add_is_major_party=party_affiliation_add_is_major_party,
    party_affiliation_add_is_institutional=party_affiliation_add_is_institutional,
    party_affiliation_add_slash_flag=party_affiliation_add_slash_flag,
    party_affiliation_add_ampersand_flag=party_affiliation_add_ampersand_flag,
    party_affiliation_add_comma_flag=party_affiliation_add_comma_flag,
    party_affiliation_add_parentheses_flag=party_affiliation_add_parentheses_flag,
    party_affiliation_add_party_primary_true_rate=party_affiliation_add_party_primary_true_rate,
    party_affiliation_party_label_col=party_affiliation_party_label_col,
    party_affiliation_scale=party_affiliation_scale, party_affiliation_verbose=party_affiliation_verbose,
    state_source_col=state_source_col, state_drop=state_drop, state_keep_original=state_keep_original,
    state_clean_text=state_clean_text, state_normalize_state=state_normalize_state,
    state_group_rare=state_group_rare, state_rare_threshold=state_rare_threshold,
    state_rare_label=state_rare_label, state_add_is_us_state=state_add_is_us_state,
    state_add_frequency=state_add_frequency, state_add_is_rare=state_add_is_rare,
    state_add_grouped_state=state_add_grouped_state, state_add_length_features=state_add_length_features,
    state_add_token_count=state_add_token_count, state_add_has_us_words=state_add_has_us_words,
    state_add_us_region=state_add_us_region, state_scale=state_scale, state_verbose=state_verbose,
    fe_statement_col=fe_statement_col, fe_statement_original_col=fe_statement_original_col,
    fe_speaker_col=fe_speaker_col, fe_subject_col=fe_subject_col, fe_party_col=fe_party_col,
    fe_speaker_job_col=fe_speaker_job_col, fe_state_col=fe_state_col, fe_label_col=fe_label_col,
    fe_add_speaker_subject=fe_add_speaker_subject, fe_add_speaker_party=fe_add_speaker_party,
    fe_add_subject_party=fe_add_subject_party, fe_add_speaker_job_subject=fe_add_speaker_job_subject,
    fe_add_state_party=fe_add_state_party,
    fe_add_speaker_statement_len_bucket=fe_add_speaker_statement_len_bucket,
    fe_statement_len_bins=fe_statement_len_bins,
    fe_add_speaker_true_rate=fe_add_speaker_true_rate, fe_add_subject_true_rate=fe_add_subject_true_rate,
    fe_add_party_true_rate=fe_add_party_true_rate,
    fe_add_speaker_avg_statement_len=fe_add_speaker_avg_statement_len,
    fe_add_subject_avg_statement_len=fe_add_subject_avg_statement_len,
    fe_add_speaker_avg_punctuation=fe_add_speaker_avg_punctuation,
    fe_add_speaker_avg_number_ratio=fe_add_speaker_avg_number_ratio,
    fe_add_negation_count=fe_add_negation_count, fe_add_hedge_count=fe_add_hedge_count,
    fe_add_absolutist_count=fe_add_absolutist_count, fe_add_numeral_count=fe_add_numeral_count,
    fe_add_proper_noun_count=fe_add_proper_noun_count, fe_add_readability=fe_add_readability,
    fe_add_sentiment=fe_add_sentiment, fe_scale=fe_scale, fe_verbose=fe_verbose,
)
print('OneStepOptions built.')

## Run Preprocessing

This is the slowest step (~10–15 min with MPNet's 768-dim encoder). Run once — all four base models share this output.

In [ ]:
print(f'[{_now()}] Running preprocessing...')
_t0 = time()
df_processed = preprocess_one_step(df, options=options)
print(f'Done in {time()-_t0:.1f}s  |  Rows: {len(df_processed):,}  |  Columns: {df_processed.shape[1]}')
df_processed.dtypes.value_counts()

## Categorical Encoding

OrdinalEncoder is applied once on the full dataset. All four base models receive the same encoded columns.

**CatBoost special case:** `dtype=int` is required. CatBoost expects integer-typed columns when passed via `cat_features` — float-encoded integers would be rejected or silently treated as numeric.

In [ ]:
_all_obj_cols = df_processed.select_dtypes(include='object').columns.tolist()
_source_cols = {
    statement_source_col, speaker_source_col, subject_source_col,
    speaker_job_source_col, party_affiliation_source_col, state_source_col,
}
_text_cols = (
    {c for c in _all_obj_cols if c.endswith(('_clean', '_original'))}
    | (_source_cols & set(_all_obj_cols))
)
_cat_cols = [c for c in _all_obj_cols if c not in _text_cols and c != label_source_col]

print(f'Text columns dropped     : {sorted(_text_cols)}')
print(f'Categorical cols encoded : {_cat_cols}')

# dtype=int required for CatBoost cat_features
ordinal_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, dtype=int)
_cat_encoded = pd.DataFrame(
    ordinal_enc.fit_transform(df_processed[_cat_cols]),
    columns=_cat_cols,
    index=df_processed.index,
)

df_features = pd.concat(
    [df_processed.select_dtypes(exclude='object'), _cat_encoded],
    axis=1,
)

# Track categorical column names for CatBoost cat_features lookup
_cat_feature_names = [c for c in _cat_encoded.columns if c != label_source_col]
print(f'\nTotal features: {df_features.shape[1]}')

## Feature Matrix and Split

In [ ]:
X = df_features.drop(columns=[label_source_col])
y = df_processed[label_source_col]

vec_cols   = [c for c in X.columns if '_vec_' in c or c.startswith('vec_')]
cat_cols_X = [c for c in X.columns if c in _cat_cols]
other_cols = [c for c in X.columns if c not in vec_cols and c not in cat_cols_X]

print(f'Embedding dimensions : {len(vec_cols)}  (MPNet 768-dim)')
print(f'Encoded categoricals : {len(cat_cols_X)}  →  {cat_cols_X}')
print(f'Other numeric        : {len(other_cols)}')
print(f'Total features       : {X.shape[1]}')
print(f'\nTarget distribution:\n{y.value_counts(normalize=True).round(4)}')

In [ ]:
X_trainval, X_holdout, y_trainval, y_holdout = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f'Train/val : {X_trainval.shape[0]:,}  |  Holdout : {X_holdout.shape[0]:,}  |  CV folds : {skf.get_n_splits()}')

## Stacking Configuration

### Base model HPs

Best-known HPs from prior experiments are used directly — **no inner HP search** in this run. This keeps training time manageable since five independent models must be trained per fold.

**Experiment D: XGBoost replaces base LR.** In earlier stacking experiments (A/B/C) a Logistic Regression was used as the first base model. Its holdout ROC-AUC was 0.6720 — the weakest in the ensemble. XGBoost, with gradient boosting and explicit regularisation, is a stronger replacement.

**XGBoost class imbalance:** XGBoost does not accept a `class_weight` dict directly. Instead, per-sample weights are passed via `sample_weight` at `.fit()` time — `{0: 1.42, 1: 0.77}` mapped to each training row.

### Meta-learner

`LogisticRegression(C=0.1)` — heavily regularised. The 4 input features are OOF probabilities (all in [0, 1]) so the meta-LR acts as a **weighted combination** of base model predictions. No class_weight is needed because the base probas already encode the imbalance.

In [ ]:
CLASS_WEIGHTS  = [1.42, 0.77]           # list form for CatBoost
CLASS_WEIGHT_D = {0: 1.42, 1: 0.77}    # dict form for RFC, LGBM
THRESHOLD      = 0.5

enable_threshold_tuning = True
overwrite_threshold     = True
THRESHOLD_METRIC        = 'macro_f1'

# drop_speaker_true_rate=True: confirmed best across individual models AND stacking
drop_speaker_true_rate    = True
enable_true_rate_features = True
true_rate_fallback        = 0.5

# Base model HPs — best known from prior experiments
BASE_XGB_HP = dict(
    n_estimators=500, learning_rate=0.03, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    eval_metric='logloss', random_state=42, n_jobs=-1, verbosity=0,
)
# XGBoost has no class_weight param — applied via sample_weight at fit time
_XGB_SW = {0: CLASS_WEIGHT_D[0], 1: CLASS_WEIGHT_D[1]}

BASE_RFC_HP = dict(
    n_estimators=500, max_features=0.3, min_samples_leaf=2,
    class_weight=CLASS_WEIGHT_D, n_jobs=-1, random_state=42,
)

BASE_LGBM_HP = dict(
    n_estimators=500, learning_rate=0.03, num_leaves=63,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.0, reg_lambda=0.0,
    class_weight=CLASS_WEIGHT_D, n_jobs=-1, random_state=42, verbose=-1,
)

BASE_CAT_HP = dict(
    iterations=300, learning_rate=0.03, depth=4, l2_leaf_reg=5,
    border_count=32, bagging_temperature=0.0,
    class_weights=CLASS_WEIGHTS, thread_count=-1, random_seed=42, verbose=0,
)

# Meta-learner: plain LR on stacked OOF probas, strongly regularised
META_LR_HP = dict(C=0.1, penalty='l2', max_iter=1000, solver='lbfgs', random_state=42)

print('Base model HPs set.')
print(f'  XGB  : {BASE_XGB_HP}')
print(f'  RFC  : {BASE_RFC_HP}')
print(f'  LGBM : {BASE_LGBM_HP}')
print(f'  CAT  : {BASE_CAT_HP}')
print(f'  Meta : {META_LR_HP}')

## Late Fusion — Optional Transformer OOF

If transformer k-fold OOF artifacts exist on disk (from running `transformer_lora_kfold_extract.py` or `transformer_kfold_base_extract.py`), they are added as a **5th base model column** in the stacking matrix.

This is called **late fusion**: the transformer is never retrained here — its pre-computed OOF probabilities are simply concatenated into the meta-learner's input. The priority order is: Mistral-7B LoRA3 > Mistral-7B LoRA > DeBERTa-v3-base > DeBERTa-v3-small.

In [ ]:
def _find_transformer_artifacts(root: Path):
    """Return (dir, oof_filename, variant_label) for the best available transformer."""
    candidates = [
        (root / 'models' / 'transformer_lora_kfold',  'mistral-7b-lora3-kfold-oof.csv', 'mistral-7b-lora3'),
        (root / 'models' / 'transformer_lora_kfold',  'mistral-7b-lora-kfold-oof.csv',  'mistral-7b-lora'),
        (root / 'models' / 'transformer_kfold_base',  'deberta-v3-base-kfold-oof.csv',   'deberta-v3-base'),
        (root / 'models' / 'transformer_kfold',       'deberta-v3-small-kfold-oof.csv',  'deberta-v3-small'),
    ]
    for d, oof_name, label in candidates:
        if (d / oof_name).exists() and (d / 'ho_proba.npy').exists() and (d / 'test_proba.npy').exists():
            return d, oof_name, label
    return None, None, None

_KFOLD_DIR, _TRANS_OOF_NAME, _TRANS_VARIANT = _find_transformer_artifacts(project_root)
use_transformer_fusion = _KFOLD_DIR is not None

if use_transformer_fusion:
    TRANSFORMER_OOF_PATH  = _KFOLD_DIR / _TRANS_OOF_NAME
    TRANSFORMER_HO_PATH   = _KFOLD_DIR / 'ho_proba.npy'
    TRANSFORMER_TEST_PATH = _KFOLD_DIR / 'test_proba.npy'
    print(f'[Late fusion] Transformer found: {_TRANS_VARIANT}')
    print(f'  OOF: {TRANSFORMER_OOF_PATH}')
else:
    print('[Late fusion] Disabled — no transformer artifacts found.')
    print('  Run transformer_kfold_base_extract.py or transformer_lora_kfold_extract.py first.')

BASE_NAMES = ['xgb', 'rfc', 'lgbm', 'cat'] + (['transformer'] if use_transformer_fusion else [])
print(f'\nBase models: {BASE_NAMES}')

## True-Rate Feature Setup

`drop_speaker_true_rate=True` is applied here — three true-rate features are included (subject, party, job), but **not speaker**. This was confirmed as the best setting across all individual models and validated again for the stacking ensemble: speaker true-rate collapses ensemble diversity by making all four base models converge on the same signal.

In [ ]:
_tr_group_cols: dict = {}
if enable_true_rate_features:
    _candidates = {
        'fe_subject_true_rate':     ['subject_primary_grouped', 'subject_primary', 'subject_clean'],
        'fe_party_true_rate':       ['party_affiliation_grouped', 'party_affiliation_clean'],
        'fe_speaker_job_true_rate': ['speaker_job_grouped', 'speaker_job_clean'],
    }
    if not drop_speaker_true_rate:
        _candidates['fe_speaker_true_rate'] = ['speaker_grouped', 'speaker_clean']

    for _feat, _src_cols in _candidates.items():
        for _col in _src_cols:
            if _col in df_processed.columns:
                _tr_group_cols[_feat] = _col
                break

    X_trainval = X_trainval.copy()
    X_holdout  = X_holdout.copy()
    for _feat in _tr_group_cols:
        X_trainval[_feat] = true_rate_fallback
        X_holdout[_feat]  = true_rate_fallback

    _grp_trainval = pd.DataFrame(
        {col: df_processed[col].loc[X_trainval.index].values for col in _tr_group_cols.values()}
    )
    _grp_trainval['_label'] = y_trainval.values

    _grp_holdout = pd.DataFrame(
        {col: df_processed[col].loc[X_holdout.index].values for col in _tr_group_cols.values()}
    )

    print(f'True-rate features : {list(_tr_group_cols.keys())}')
    print(f'Source columns     : {list(_tr_group_cols.values())}')

## Cross-Validation — Collect OOF Probabilities

The stacking CV loop trains **all four base models per fold** and stores their validation-fold predictions into four OOF arrays. After all 5 folds, each array covers all 7,160 train/val rows without overlap.

Key points:
- True-rate features are recomputed fold-safe for every base model (same lookup, same fold split).
- CatBoost needs `cat_features` indices recomputed each fold because true-rate columns are appended after OrdinalEncoded columns, shifting the index positions.
- Per-fold metrics use **simple average** of the four OOF probas as a quick proxy — the meta-LR is trained later on the full OOF matrix.
- XGBoost uses `sample_weight` (not `class_weight`) for imbalance correction.

In [ ]:
print(f'[{_now()}] Starting cross-validation...')
_cv_start = time()

# One OOF array per base model
oof_xgb  = np.zeros(len(X_trainval))
oof_rfc  = np.zeros(len(X_trainval))
oof_lgbm = np.zeros(len(X_trainval))
oof_cat  = np.zeros(len(X_trainval))
oof_true = np.zeros(len(X_trainval), dtype=int)

cv_fold_metrics = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_trainval, y_trainval), 1):
    _fold_t = time()

    X_fold_train = X_trainval.iloc[train_idx].copy()
    X_fold_val   = X_trainval.iloc[val_idx].copy()
    y_fold_train = y_trainval.iloc[train_idx]
    y_fold_val   = y_trainval.iloc[val_idx]

    # True-rate features — fold-safe, applied identically to all 4 base models
    if enable_true_rate_features and _tr_group_cols:
        _grp_tr = _grp_trainval.iloc[train_idx]
        _grp_vl = _grp_trainval.iloc[val_idx]
        for _feat, _src_col in _tr_group_cols.items():
            _rate_map = _grp_tr.groupby(_src_col)['_label'].mean()
            X_fold_train[_feat] = _grp_tr[_src_col].map(_rate_map).fillna(true_rate_fallback).values
            X_fold_val[_feat]   = _grp_vl[_src_col].map(_rate_map).fillna(true_rate_fallback).values

    # XGBoost — class imbalance via sample_weight (no class_weight param)
    _xgb_sw = np.where(y_fold_train == 0, _XGB_SW[0], _XGB_SW[1])
    _m_xgb  = XGBClassifier(**BASE_XGB_HP)
    _m_xgb.fit(X_fold_train, y_fold_train, sample_weight=_xgb_sw)
    oof_xgb[val_idx] = _m_xgb.predict_proba(X_fold_val)[:, 1]

    # RFC
    _m_rfc = RandomForestClassifier(**BASE_RFC_HP)
    _m_rfc.fit(X_fold_train, y_fold_train)
    oof_rfc[val_idx] = _m_rfc.predict_proba(X_fold_val)[:, 1]

    # LightGBM
    _m_lgbm = LGBMClassifier(**BASE_LGBM_HP)
    _m_lgbm.fit(X_fold_train, y_fold_train)
    oof_lgbm[val_idx] = _m_lgbm.predict_proba(X_fold_val)[:, 1]

    # CatBoost — cat_features indices recomputed per fold (true-rate columns shift positions)
    _cat_indices = [X_fold_train.columns.get_loc(c) for c in _cat_feature_names
                    if c in X_fold_train.columns]
    _m_cat = CatBoostClassifier(**BASE_CAT_HP, cat_features=_cat_indices)
    _m_cat.fit(X_fold_train, y_fold_train)
    oof_cat[val_idx] = _m_cat.predict_proba(X_fold_val)[:, 1]

    oof_true[val_idx] = y_fold_val.values

    # Quick per-fold proxy: simple average of 4 OOF probas
    _meta_avg  = np.column_stack([oof_xgb[val_idx], oof_rfc[val_idx],
                                   oof_lgbm[val_idx], oof_cat[val_idx]]).mean(axis=1)
    _preds_avg = (_meta_avg >= 0.5).astype(int)

    fold_metrics = {
        'fold':         fold_idx,
        'roc_auc_avg':  roc_auc_score(y_fold_val, _meta_avg),
        'macro_f1_avg': f1_score(y_fold_val, _preds_avg, average='macro', zero_division=0),
        'roc_auc_xgb':  roc_auc_score(y_fold_val, oof_xgb[val_idx]),
        'roc_auc_rfc':  roc_auc_score(y_fold_val, oof_rfc[val_idx]),
        'roc_auc_lgbm': roc_auc_score(y_fold_val, oof_lgbm[val_idx]),
        'roc_auc_cat':  roc_auc_score(y_fold_val, oof_cat[val_idx]),
    }
    cv_fold_metrics.append(fold_metrics)

    print(
        f'  Fold {fold_idx} | avg-ROC={fold_metrics["roc_auc_avg"]:.4f}  '
        f'avg-F1={fold_metrics["macro_f1_avg"]:.4f}  '
        f'XGB={fold_metrics["roc_auc_xgb"]:.4f}  '
        f'RFC={fold_metrics["roc_auc_rfc"]:.4f}  '
        f'LGBM={fold_metrics["roc_auc_lgbm"]:.4f}  '
        f'CAT={fold_metrics["roc_auc_cat"]:.4f}  '
        f'({time()-_fold_t:.1f}s)'
    )

print(f'\nTotal CV time: {time()-_cv_start:.1f}s')

## CV Summary — Per-Model OOF ROC-AUC

The per-model OOF ROC-AUC shows the **individual contribution of each base model** before the meta-LR combines them. Lower individual AUC is not necessarily bad — a weaker but less correlated model can still add ensemble value.

In [ ]:
cv_keys = ['roc_auc_avg', 'macro_f1_avg', 'roc_auc_xgb', 'roc_auc_rfc', 'roc_auc_lgbm', 'roc_auc_cat']
summary_rows = {}
for k in cv_keys:
    vals = [m[k] for m in cv_fold_metrics]
    summary_rows[k] = {'mean': float(np.mean(vals)), 'std': float(np.std(vals))}

summary_df = pd.DataFrame(summary_rows).T.round(4)
print('Cross-validation summary (5-fold):')
display(summary_df)

folds_df = pd.DataFrame(cv_fold_metrics).set_index('fold').round(4)
print('\nPer-fold results:')
display(folds_df)

# Visualise per-model OOF ROC-AUC across folds
fig, ax = plt.subplots(figsize=(9, 4))
folds = [m['fold'] for m in cv_fold_metrics]
for model, col, linestyle in [
    ('XGB',  'roc_auc_xgb',  '-o'),
    ('RFC',  'roc_auc_rfc',  '-s'),
    ('LGBM', 'roc_auc_lgbm', '-^'),
    ('CAT',  'roc_auc_cat',  '-D'),
    ('Avg',  'roc_auc_avg',  '--k'),
]:
    vals = [m[col] for m in cv_fold_metrics]
    ax.plot(folds, vals, linestyle, label=f'{model} (mean={np.mean(vals):.4f})', markersize=5)
ax.set_xlabel('Fold')
ax.set_ylabel('ROC-AUC')
ax.set_title('Per-Model OOF ROC-AUC per Fold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Late Fusion — Load Transformer OOF

If transformer artifacts were found, load the OOF CSV and align its rows to `X_trainval`'s index order. The CSV stores `(idx, oof_proba)` pairs using the original DataFrame index — a dictionary lookup maps each training row to its pre-computed probability.

In [ ]:
if use_transformer_fusion:
    print(f'[{_now()}] Loading transformer OOF...')
    _trans_oof_df = pd.read_csv(TRANSFORMER_OOF_PATH)
    _idx_to_proba = dict(zip(_trans_oof_df['idx'].astype(int), _trans_oof_df['oof_proba']))
    oof_transformer = np.array([_idx_to_proba[i] for i in X_trainval.index], dtype=np.float64)

    trans_roc = roc_auc_score(y_trainval, oof_transformer)
    print(f'  Loaded {len(_trans_oof_df):,} OOF rows  '
          f'range=[{oof_transformer.min():.3f}, {oof_transformer.max():.3f}]')
    print(f'  Transformer OOF ROC-AUC: {trans_roc:.4f}')
else:
    print('Transformer late fusion disabled — skipping.')

## Train Meta-LR on Stacked OOF

The meta-learner sees only the stacked OOF probabilities as input — a matrix of shape `(N_trainval, 4)` or `(N_trainval, 5)` with transformer.

**Why Logistic Regression?** The 4 input features are all probabilities in [0, 1], and their relationship to the true label is expected to be roughly monotone. LR is the natural choice: it learns a weighted blend without overfitting on a 4-feature dataset. High regularisation (`C=0.1`) prevents any single base model from being assigned an extreme coefficient.

In [ ]:
print(f'[{_now()}] Training meta-LR on stacked OOF...')
_oof_cols = [oof_xgb, oof_rfc, oof_lgbm, oof_cat]
if use_transformer_fusion:
    _oof_cols.append(oof_transformer)

meta_X_train = np.column_stack(_oof_cols)
print(f'  Meta-LR input shape: {meta_X_train.shape}  (rows × base_models)')

meta_lr = LogisticRegression(**META_LR_HP)
meta_lr.fit(meta_X_train, y_trainval)

meta_coefs = dict(zip(BASE_NAMES, meta_lr.coef_[0].tolist()))
print(f'\nMeta-LR coefficients (positive = higher proba → more likely false):')
for name, coef in sorted(meta_coefs.items(), key=lambda x: -x[1]):
    bar = '█' * int(abs(coef) * 20)
    print(f'  {name:15s}: {coef:+.4f}  {bar}')

## Threshold Tuning on Meta-LR OOF Probabilities

The threshold is tuned on the meta-LR's own probability predictions over the full OOF matrix. This is analogous to tuning on individual model OOF probas, but now the probas come from the second level — they already incorporate the ensemble's learned weighting.

In [ ]:
oof_meta_proba = meta_lr.predict_proba(meta_X_train)[:, 1]

if enable_threshold_tuning:
    _metric_fn = {
        'macro_f1':     lambda t, yt, yp: f1_score(yt, yp, average='macro', zero_division=0),
        'mcc':          lambda t, yt, yp: matthews_corrcoef(yt, yp),
        'balanced_acc': lambda t, yt, yp: balanced_accuracy_score(yt, yp),
    }[THRESHOLD_METRIC]

    threshold_grid   = np.arange(0.20, 0.76, 0.01)
    threshold_scores = {}
    for t in threshold_grid:
        preds = (oof_meta_proba >= t).astype(int)
        threshold_scores[round(float(t), 2)] = _metric_fn(t, oof_true, preds)

    best_threshold = max(threshold_scores, key=threshold_scores.get)
    best_score     = threshold_scores[best_threshold]

    fig, ax = plt.subplots(figsize=(9, 4))
    ts = list(threshold_scores.keys())
    ss = list(threshold_scores.values())
    ax.plot(ts, ss, marker='o', markersize=3, linewidth=1.5, label=THRESHOLD_METRIC)
    ax.axvline(best_threshold, color='red', linestyle='--', alpha=0.7,
               label=f'Best = {best_threshold:.2f} ({best_score:.4f})')
    ax.axvline(0.5, color='gray', linestyle=':', alpha=0.6, label='Default = 0.50')
    ax.set_xlabel('Threshold')
    ax.set_ylabel(THRESHOLD_METRIC)
    ax.set_title('Threshold Sweep on Meta-LR OOF Probabilities')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f'Best threshold: {best_threshold:.2f}  (OOF {THRESHOLD_METRIC}={best_score:.4f})')

    if overwrite_threshold:
        print(f'THRESHOLD updated: {THRESHOLD:.2f} → {best_threshold:.2f}')
        THRESHOLD = best_threshold

## Final Fit — Retrain All Base Models on Full Train/Val

The meta-LR is already trained (on the OOF matrix). Now the base models are retrained on **all 7,160 rows** — no held-out fold — so their final predictions on test data use all available training signal.

In [ ]:
print(f'[{_now()}] Fitting final base models on full train/val set...')
_t0 = time()

_final_rate_maps: dict = {}
X_trainval_final = X_trainval.copy()
X_holdout_final  = X_holdout.copy()

if enable_true_rate_features and _tr_group_cols:
    for _feat, _src_col in _tr_group_cols.items():
        _rate_map = _grp_trainval.groupby(_src_col)['_label'].mean()
        X_trainval_final[_feat] = _grp_trainval[_src_col].map(_rate_map).fillna(true_rate_fallback).values
        X_holdout_final[_feat]  = _grp_holdout[_src_col].map(_rate_map).fillna(true_rate_fallback).values
        _final_rate_maps[_feat] = _rate_map.to_dict()

# XGBoost
_xgb_sw_final = np.where(np.array(y_trainval) == 0, _XGB_SW[0], _XGB_SW[1])
final_xgb = XGBClassifier(**BASE_XGB_HP)
final_xgb.fit(X_trainval_final, y_trainval, sample_weight=_xgb_sw_final)
print('  XGBoost done')

# RFC
final_rfc = RandomForestClassifier(**BASE_RFC_HP)
final_rfc.fit(X_trainval_final, y_trainval)
print('  RFC done')

# LightGBM
final_lgbm = LGBMClassifier(**BASE_LGBM_HP)
final_lgbm.fit(X_trainval_final, y_trainval)
print('  LightGBM done')

# CatBoost
_cat_indices_final = [X_trainval_final.columns.get_loc(c) for c in _cat_feature_names
                      if c in X_trainval_final.columns]
final_cat = CatBoostClassifier(**BASE_CAT_HP, cat_features=_cat_indices_final)
final_cat.fit(X_trainval_final, y_trainval)
print('  CatBoost done')

print(f'  Total: {time()-_t0:.1f}s')

## Holdout Evaluation

The holdout set is touched for the first time. Each base model predicts on holdout, then the predictions are stacked and passed through the meta-LR. If transformer artifacts exist, the pre-saved `ho_proba.npy` is loaded and aligned via index lookup.

In [ ]:
print(f'Using threshold: {THRESHOLD:.2f}\n')

h_xgb  = final_xgb.predict_proba(X_holdout_final)[:, 1]
h_rfc  = final_rfc.predict_proba(X_holdout_final)[:, 1]
h_lgbm = final_lgbm.predict_proba(X_holdout_final)[:, 1]
h_cat  = final_cat.predict_proba(X_holdout_final)[:, 1]

_ho_cols = [h_xgb, h_rfc, h_lgbm, h_cat]
if use_transformer_fusion:
    _ho_idx_kfold    = np.load(_KFOLD_DIR / 'ho_idx.npy')
    _ho_trans_lookup = dict(zip(_ho_idx_kfold.tolist(), np.load(TRANSFORMER_HO_PATH).tolist()))
    _h_transformer   = np.array([_ho_trans_lookup[i] for i in X_holdout.index])
    _ho_cols.append(_h_transformer)
    print(f'  Transformer holdout range=[{_h_transformer.min():.3f}, {_h_transformer.max():.3f}]')

meta_X_holdout = np.column_stack(_ho_cols)
y_proba = meta_lr.predict_proba(meta_X_holdout)[:, 1]
y_pred  = (y_proba >= THRESHOLD).astype(int)

holdout_metrics = {
    'roc_auc':      roc_auc_score(y_holdout, y_proba),
    'pr_auc':       average_precision_score(y_holdout, y_proba),
    'macro_f1':     f1_score(y_holdout, y_pred, average='macro', zero_division=0),
    'f1_class1':    f1_score(y_holdout, y_pred, zero_division=0),
    'precision':    precision_score(y_holdout, y_pred, zero_division=0),
    'recall':       recall_score(y_holdout, y_pred, zero_division=0),
    'accuracy':     accuracy_score(y_holdout, y_pred),
    'mcc':          matthews_corrcoef(y_holdout, y_pred),
    'balanced_acc': balanced_accuracy_score(y_holdout, y_pred),
}
cm     = confusion_matrix(y_holdout, y_pred)
report = classification_report(y_holdout, y_pred, output_dict=True)

metrics_df = pd.DataFrame.from_dict(holdout_metrics, orient='index', columns=['value']).round(4)
display(metrics_df)
print()
print(classification_report(y_holdout, y_pred, target_names=['True (0)', 'False (1)']))

# Individual base model holdout ROC-AUC for comparison
print('\nBase model holdout ROC-AUC (for reference):')
base_rocs = {
    'XGBoost':   roc_auc_score(y_holdout, h_xgb),
    'RFC':       roc_auc_score(y_holdout, h_rfc),
    'LightGBM':  roc_auc_score(y_holdout, h_lgbm),
    'CatBoost':  roc_auc_score(y_holdout, h_cat),
    'Stack':     holdout_metrics['roc_auc'],
}
for name, auc in base_rocs.items():
    marker = '  ← ensemble' if name == 'Stack' else ''
    print(f'  {name:12s}: {auc:.4f}{marker}')

## Evaluation Plots

In [ ]:
fpr, tpr, _      = roc_curve(y_holdout, y_proba)
prec_c, rec_c, _ = precision_recall_curve(y_holdout, y_proba)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC
axes[0].plot(fpr, tpr, label=f"Stack ROC-AUC = {holdout_metrics['roc_auc']:.4f}")
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.6)
axes[0].set_title('ROC Curve — Stacking (holdout)')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# PR
axes[1].plot(rec_c, prec_c, label=f"PR-AUC = {holdout_metrics['pr_auc']:.4f}")
axes[1].set_title('Precision-Recall Curve (holdout)')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Confusion matrix
im = axes[2].imshow(cm, interpolation='nearest', cmap='Blues')
axes[2].set_title(f'Confusion Matrix (threshold={THRESHOLD:.2f})')
axes[2].set_xticks([0, 1])
axes[2].set_yticks([0, 1])
axes[2].set_xticklabels(['Pred: True (0)', 'Pred: False (1)'])
axes[2].set_yticklabels(['Actual: True (0)', 'Actual: False (1)'])
for i in range(2):
    for j in range(2):
        axes[2].text(j, i, str(cm[i, j]), ha='center', va='center',
                     color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=14)
fig.colorbar(im, ax=axes[2])

plt.suptitle(
    f'Stacking Ensemble — Holdout  |  macro_f1={holdout_metrics["macro_f1"]:.4f}  '
    f'ROC-AUC={holdout_metrics["roc_auc"]:.4f}  threshold={THRESHOLD:.2f}',
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.show()

## Base Model Comparison Plot

Visualise individual vs ensemble holdout ROC-AUC to see whether stacking actually improved over the best individual model.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
names = list(base_rocs.keys())
aucs  = list(base_rocs.values())
colors = ['steelblue'] * 4 + ['tomato']
bars = ax.barh(names, aucs, color=colors)
ax.set_xlabel('ROC-AUC (holdout)')
ax.set_title('Base Models vs Stacking Ensemble — Holdout ROC-AUC')
ax.set_xlim(min(aucs) - 0.02, max(aucs) + 0.02)
for bar, auc in zip(bars, aucs):
    ax.text(auc + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{auc:.4f}', va='center', fontsize=9)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## Summary

| Metric | Holdout |
|---|---|
| Macro F1 | (see above) |
| ROC-AUC | (see above) |
| MCC | (see above) |
| Balanced Accuracy | (see above) |

**Why stacking did not dominate the competition:**

The base models (XGB, RFC, LGBM, CatBoost) all see the same features and share most of the same signal (speaker true-rate, embeddings). When base models are highly correlated, stacking gains little — the meta-LR has nothing new to learn. In the project results, the stacking macro F1 was **0.6168** vs CatBoost-optB alone at a similar level. The gap vs transformers (0.73+ ROC-AUC) showed that structured feature engineering had hit its ceiling.

**Key design decisions revisited:**
- `drop_speaker_true_rate=True` — prevents all base models from collapsing to the same dominant signal
- XGBoost replaces base LR (Experiment D) — stronger individual contribution
- Meta-LR `C=0.1` — prevents extreme weighting of one base model
- Late fusion with transformer OOF — the transformer brings a qualitatively different signal (dense LM-based representations) that is much less correlated with the GBDT models